# QC participant assignments per rater

Summarizes the participants each rater is assigned in the rater config `RATER_CONFIG`
(QC-Studio's multi-user config format, see `ui/utils/user_config.py`):

1. **Config check** – valid vs rejected raters, missing participant lists.
2. **Per-rater counts** – rows, unique participants, sessions, duplicates, shared vs exclusive participants.
3. **Coverage across raters** – how many raters see each participant.
4. **Pairwise overlaps** – shared-participant counts and Jaccard index between every pair of raters.
5. **Overlap participants** – the participants rated by more than one rater (inter-rater pool).
6. **New rater list** – builds a participant list for a new rater from the least-covered participants.

Paths are resolved with the app's own `load_users_config`, so they match what each rater sees in QC-Studio.

In [ ]:
# --- Local paths --------------------------------------------------------------
# Machine-specific paths live in notebooks/local_config.json (gitignored).
# Create it by copying notebooks/local_config.example.json and editing the paths.
import json
from pathlib import Path

LOCAL_CONFIG = next(
    (p for p in (Path("local_config.json"), Path("notebooks") / "local_config.json") if p.is_file()), None
)
if LOCAL_CONFIG is None:
    raise FileNotFoundError(
        "notebooks/local_config.json not found: copy notebooks/local_config.example.json and set your paths"
    )
_local = json.loads(LOCAL_CONFIG.read_text())
print(f"Local config: {LOCAL_CONFIG.resolve()}")

RATER_CONFIG = _local["rater_config"]            # every rater, current and new
PARTICIPANT_LIST = _local["participant_list"]    # dataset-level participant pool (used by section 8)
OUTPUT_DIR = _local.get("output_dir")            # write summary TSVs here (None = don't save)

# --- Settings ---------------------------------------------------------------
# Overlap is computed on participant_id only, or on (participant_id, session_id)
# when every list has a session_id column. Force participant-level with True.
PARTICIPANT_LEVEL_ONLY = False


In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import pandas as pd

# Reuse the app's config parser so path resolution matches QC-Studio exactly.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ui" / "utils" / "user_config.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "ui"))
from utils.user_config import load_users_config 

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 1. Load rater config

In [ ]:
cfg = load_users_config(RATER_CONFIG)
print(f"Config:     {cfg.path}")
print(f"Rater root: {cfg.rater_root}")
print(f"Valid raters: {len(cfg.users)}   Rejected: {len(cfg.rejected)}")

raters_df = pd.DataFrame(cfg.users.values())[
    ["rater_id", "email", "participant_list", "session_list", "out_dir"]
].sort_values("rater_id").reset_index(drop=True)
raters_df["list_exists"] = raters_df["participant_list"].map(lambda p: Path(p).is_file())
raters_df["list_file"] = raters_df["participant_list"].map(lambda p: Path(p).name)
display(raters_df[["rater_id", "email", "list_file", "list_exists", "session_list"]])

if cfg.rejected:
    print("\nRejected entries (these raters cannot sign in):")
    for email, reasons in cfg.rejected.items():
        print(f"  {email}: " + "; ".join(reasons))
missing = raters_df.loc[~raters_df["list_exists"], ["rater_id", "participant_list"]]
if not missing.empty:
    print("\nMissing participant lists (excluded below):")
    display(missing)

## 2. Load participant lists

In [ ]:
frames = []
for u in raters_df.itertuples():
    if not u.list_exists:
        continue
    df = pd.read_csv(u.participant_list, sep="\t", dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    if "participant_id" not in df.columns:
        print(f"WARNING {u.rater_id}: no participant_id column in {u.participant_list}; skipped")
        continue
    keep = ["participant_id"] + (["session_id"] if "session_id" in df.columns else [])
    df = df[keep].apply(lambda s: s.str.strip())
    df = df[df["participant_id"] != ""]
    df.insert(0, "rater_id", u.rater_id)
    frames.append(df)

assign = pd.concat(frames, ignore_index=True)
has_sessions = "session_id" in assign.columns and assign["session_id"].notna().all()
KEY = ["participant_id"] if (PARTICIPANT_LEVEL_ONLY or not has_sessions) else ["participant_id", "session_id"]
assign["unit"] = assign[KEY].agg("|".join, axis=1)
print(f"Overlap unit: {' + '.join(KEY)}")
print(f"Assignments loaded: {len(assign)} rows across {assign['rater_id'].nunique()} raters")

## 3. Per-rater counts

In [ ]:
# How many raters each unit (participant or participant+session) is assigned to
unit_raters = assign.drop_duplicates(["rater_id", "unit"]).groupby("unit")["rater_id"].nunique()
assign["n_raters_for_unit"] = assign["unit"].map(unit_raters)

def summarize(g):
    uniq = g.drop_duplicates("unit")
    return pd.Series({
        "rows": len(g),
        "unique_participants": g["participant_id"].nunique(),
        "unique_units": len(uniq),
        "duplicate_rows": len(g) - len(uniq),
        "n_sessions": g["session_id"].nunique() if "session_id" in g else 0,
        "shared_with_others": int((uniq["n_raters_for_unit"] > 1).sum()),
        "exclusive": int((uniq["n_raters_for_unit"] == 1).sum()),
    })

per_rater = assign.groupby("rater_id").apply(summarize, include_groups=False).astype(int)
per_rater["pct_shared"] = (100 * per_rater["shared_with_others"] / per_rater["unique_units"]).round(1)
per_rater = per_rater.sort_values("unique_units", ascending=False)
display(per_rater)
print(per_rater[["unique_units", "shared_with_others", "exclusive"]].describe().round(1))

## 4. Coverage across raters

How many distinct participants are assigned overall, and how many raters each one gets.

In [ ]:
total_units = len(unit_raters)
print(f"Distinct {' + '.join(KEY)} across all raters: {total_units}")
print(f"Distinct participants: {assign['participant_id'].nunique()}")
print(f"Sum of per-rater assignments: {int(per_rater['unique_units'].sum())}")

coverage = (
    unit_raters.value_counts().sort_index()
    .rename_axis("n_raters").rename("n_units").to_frame()
)
coverage["pct"] = (100 * coverage["n_units"] / total_units).round(1)
coverage["cum_pct_at_least"] = (100 * coverage["n_units"][::-1].cumsum()[::-1] / total_units).round(1)
display(coverage)

## 5. Pairwise overlaps between raters

Diagonal = each rater's unique units. Off-diagonal = units shared by the pair.

In [ ]:
sets = assign.groupby("rater_id")["unit"].agg(set)
raters = sorted(sets.index)

overlap = pd.DataFrame(0, index=raters, columns=raters, dtype=int)
jaccard = pd.DataFrame(0.0, index=raters, columns=raters)
for a in raters:
    for b in raters:
        inter = len(sets[a] & sets[b])
        overlap.loc[a, b] = inter
        jaccard.loc[a, b] = inter / len(sets[a] | sets[b]) if (sets[a] | sets[b]) else 0.0

print("Shared unit counts:")
display(overlap)

In [ ]:
# Pairs ranked by overlap (only pairs that share something)
pairs = pd.DataFrame(
    [
        {
            "rater_a": a, "rater_b": b,
            "shared": len(sets[a] & sets[b]),
            "n_a": len(sets[a]), "n_b": len(sets[b]),
            "jaccard": round(jaccard.loc[a, b], 3),
            "identical": sets[a] == sets[b],
        }
        for a, b in combinations(raters, 2)
    ]
).sort_values(["shared", "jaccard"], ascending=False)
pairs = pairs[pairs["shared"] > 0].reset_index(drop=True)
print(f"{len(pairs)} of {len(raters) * (len(raters) - 1) // 2} rater pairs share at least one unit")
if pairs["identical"].any():
    print("WARNING: some raters have identical lists:")
    display(pairs[pairs["identical"]])
display(pairs.head())

In [ ]:
# Optional heatmap (skipped if matplotlib isn't installed)
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed; skipping heatmap")
else:
    fig, ax = plt.subplots(figsize=(0.5 * len(raters) + 3, 0.5 * len(raters) + 2))
    im = ax.imshow(jaccard.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(raters)), raters, rotation=90)
    ax.set_yticks(range(len(raters)), raters)
    for i in range(len(raters)):
        for j in range(len(raters)):
            ax.text(j, i, overlap.iat[i, j], ha="center", va="center", fontsize=7,
                    color="white" if jaccard.iat[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, label="Jaccard")
    ax.set_title("Rater overlap (cell text = shared count)")
    plt.tight_layout()
    plt.show()

## 6. Participants assigned to more than one rater

The inter-rater reliability pool.

In [ ]:
display(assign["n_raters_for_unit"].value_counts())

assign.head()

## 8. Create a participant list for a new rater

Builds the list for **one** new rater from the participants assigned to the **fewest raters**.

* **Assignments come from the rater config** (`RATER_CONFIG`, which must list every rater, current and new): each rater's assignments are
  the `participant_list` it points to. A rater whose list doesn't exist yet (a new rater) has no assignments.
  Set `LIST_GLOB` (e.g. `"*_participant_list.tsv"`) to also count every matching list in each configured rater's
  folder, such as secondary lists.
* `rank_participants_by_rater_count(rater_config, pool_ids)` gives every participant in the pool its number of raters
  (a rater counts once even if several of their lists include the participant), sorted fewest first, with ties shuffled using `SEED`.
* `create_rater_list(rater_id, n, pool_ids, assignments)` takes the `n` participants with the fewest raters. Any existing
  rows for `rater_id` are ignored first, so re-running for the same rater gives the same list.
* `write_participant_list` writes to the `participant_list` path that the rater config gives that rater, only when
  `WRITE = True`, and never replaces an existing file unless `OVERWRITE = True`. The config then needs no edits.
* For several new raters, run this once per rater and write each list before the next run, so the next rater moves on
  to the next-least-covered participants.

In [ ]:
# --- Settings for the new list -------------------------------------------------
# RATER_CONFIG (every rater, current and new) and PARTICIPANT_LIST (global pool) come from local_config.json
NEW_RATER = "nikhil"                           # rater_id in RATER_CONFIG
N_PARTICIPANTS = 100                             # how many participants to assign
LIST_GLOB = None                                 # None = only each rater's participant_list; or e.g. "*_participant_list.tsv"
SEED = 42
WRITE = False
OVERWRITE = False

In [ ]:
import numpy as np

def read_participant_ids(path):
    """Unique, non-blank participant_id values from a TSV."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    if "participant_id" not in df.columns:
        raise ValueError(f"{path}: no participant_id column")
    ids = df["participant_id"].str.strip()
    return ids[ids != ""].unique().tolist()


def load_assignments(rater_config, list_glob=None, verbose=True):
    """Existing assignments of every rater in a rater config, as a long table: rater_id, list_file, participant_id.

    Uses each rater's participant_list; with list_glob, every matching file in the rater's rater_dir as well.
    Raters whose list doesn't exist yet are reported and contribute no rows.
    """
    ucfg = load_users_config(rater_config)
    if ucfg.rejected:
        print(f"WARNING: rejected entries in {rater_config} are ignored: {sorted(ucfg.rejected)}")
    rows, no_list = [], []
    for u in ucfg.users.values():
        files = {Path(u["participant_list"])}
        if list_glob:
            files |= set(Path(u["rater_dir"]).glob(list_glob))
        files = sorted(f for f in files if f.is_file())
        if not files:
            no_list.append(u["rater_id"])
        for f in files:
            rows += [(u["rater_id"], f.name, pid) for pid in read_participant_ids(f)]
    if verbose and no_list:
        print(f"Raters without an existing list (no assignments): {', '.join(sorted(no_list))}")
    return pd.DataFrame(rows, columns=["rater_id", "list_file", "participant_id"])


def _rank(pool_ids, assignments, seed=None):
    """Rank the pool by number of raters in `assignments` (fewest first, ties shuffled with seed)."""
    pool = pd.Index(pd.unique(pd.Series(list(pool_ids), dtype=str).str.strip()))
    a = assignments.drop_duplicates(["participant_id", "rater_id"])
    outside = set(a["participant_id"]) - set(pool)
    if outside:
        print(f"WARNING: {len(outside)} assigned participants are not in the pool and are ignored, e.g. {sorted(outside)[:5]}")
    per_pid = a.groupby("participant_id")["rater_id"]
    ranked = pd.DataFrame({
        "participant_id": pool,
        "n_raters": per_pid.nunique().reindex(pool, fill_value=0).astype(int).values,
        "raters": per_pid.agg(lambda s: ", ".join(sorted(s))).reindex(pool, fill_value="").values,
    })
    ranked["_tiebreak"] = np.random.default_rng(seed).random(len(ranked))
    return ranked.sort_values(["n_raters", "_tiebreak"]).drop(columns="_tiebreak").reset_index(drop=True)


def rank_participants_by_rater_count(rater_config, pool_ids, seed=None, list_glob=None, exclude_raters=()):
    """Every participant in the pool with its number of assigned raters, fewest first.

    rater_config: path to a rater config (e.g. users_all.json); assignments are read from its raters' lists.
    pool_ids: iterable of participant IDs (the global pool).
    exclude_raters: rater_ids to leave out of the counts (e.g. the rater you are about to (re)create).
    Returns a DataFrame: participant_id, n_raters, raters.
    """
    assignments = load_assignments(rater_config, list_glob)
    assignments = assignments[~assignments["rater_id"].isin(exclude_raters)]
    return _rank(pool_ids, assignments, seed)


def create_rater_list(rater_id, n, pool_ids, assignments, seed=None):
    """Participant list for one new rater: the n participants with the fewest raters.

    pool_ids: global participant pool. assignments: DataFrame with rater_id and participant_id columns
    (e.g. from load_assignments). Existing rows for rater_id are ignored, so re-running gives the same list.
    Returns the chosen rows (participant_id, n_raters before this rater, raters), sorted by participant ID.
    """
    ranked = _rank(pool_ids, assignments[assignments["rater_id"] != rater_id], seed)
    if n > len(ranked):
        raise ValueError(f"{rater_id}: asked for {n} participants but the pool has only {len(ranked)}")
    chosen = ranked.head(n)
    ids = chosen["participant_id"]
    order = sorted(range(len(chosen)), key=lambda i: (len(ids.iat[i]), ids.iat[i]))
    return chosen.iloc[order].reset_index(drop=True)


def write_participant_list(participant_ids, rater_config, rater_id, overwrite=False):
    """Write the list to the participant_list path the rater config gives rater_id; returns (path, status)."""
    raters = {u["rater_id"]: u for u in load_users_config(rater_config).users.values()}
    if rater_id not in raters:
        raise KeyError(f"{rater_id!r} is not a valid rater in {rater_config}")
    out = Path(raters[rater_id]["participant_list"])
    if out.exists() and not overwrite:
        return out, "exists, not written (set OVERWRITE=True)"
    out.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"participant_id": list(participant_ids)}).to_csv(out, sep="\t", index=False)
    return out, "written"

In [ ]:
pool_ids = read_participant_ids(PARTICIPANT_LIST)
ranked = rank_participants_by_rater_count(RATER_CONFIG, pool_ids, seed=SEED, list_glob=LIST_GLOB,
                                          exclude_raters=[NEW_RATER])
print(f"Pool: {len(pool_ids)} participants")
print("Raters per participant (before):")
display(ranked["n_raters"].value_counts().sort_index().rename_axis("n_raters").rename("n_participants").to_frame())
display(ranked.head(10))

In [ ]:
assignments = load_assignments(RATER_CONFIG, LIST_GLOB, verbose=False)
new_list = create_rater_list(NEW_RATER, N_PARTICIPANTS, pool_ids, assignments, seed=SEED)
print(f"{NEW_RATER}: {len(new_list)} participants; raters they had before:")
display(new_list["n_raters"].value_counts().sort_index().rename_axis("prior_n_raters").rename("n_participants").to_frame())
display(new_list.head())

if WRITE:
    out, status = write_participant_list(new_list["participant_id"], RATER_CONFIG, NEW_RATER, overwrite=OVERWRITE)
    print(f"{out}: {status}")
else:
    print("Dry run: set WRITE = True to save the list")